In [3]:
import os
import mesa
import time
import math
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

# 1. The Core Math: Cosine Similarity Engine
def cosine_similarity(v1, v2):
    dot_product = sum(a * b for a, b in zip(v1, v2))
    magnitude1 = math.sqrt(sum(a * a for a in v1))
    magnitude2 = math.sqrt(sum(b * b for b in v2))
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)

class RAGAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.memory_db = [] # Stores dictionaries: {"text": string, "vector": list[float]}

    # 2. The Embedding Bridge: Convert English to Math
    def embed_text(self, text):
        response = client.models.embed_content(
            model='gemini-embedding-001',
            contents=text
        )
        return response.embeddings[0].values

    def add_memory(self, text):
        vector = self.embed_text(text)
        self.memory_db.append({"text": text, "vector": vector})
        print(f"🧠 [MEMORY STORED] {text}")

    # 3. The Retrieval Engine
    def retrieve_relevant_memories(self, query, top_k=1):
        if not self.memory_db:
            return []
        
        query_vector = self.embed_text(query)
        
        scored_memories = []
        for memory in self.memory_db:
            sim = cosine_similarity(query_vector, memory["vector"])
            scored_memories.append((sim, memory["text"]))
        
        scored_memories.sort(key=lambda x: x[0], reverse=True)
        
        return [mem[1] for mem in scored_memories[:top_k]]

    def step(self):
        current_situation = "I am starving. I see a large bush covered in bright red berries. What should I do?"
        print(f"\n👀 [SITUATION ENCOUNTERED] {current_situation}")
        
        # 4. Agent queries its own brain before acting
        relevant_memories = self.retrieve_relevant_memories(current_situation, top_k=1)
        memory_context = "\n".join(relevant_memories) if relevant_memories else "No prior relevant memories."
        print(f"🔍 [RECALLED MEMORY] {memory_context}")
        
        prompt = f"""
        You are a survivalist. 
        Current Situation: {current_situation}
        Relevant Past Memory: {memory_context}
        
        Based ONLY on your past memory, what is your exact next move? Respond in one concise sentence.
        """
        
        try:
            # 5. Final Generation based on retrieved context
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt
            )
            print(f"🤖 [DECISION] {response.text.strip()}")
            
        except Exception as e:
            print(f"Error: {e}")
        
        time.sleep(4)

class RAGModel(mesa.Model):
    def __init__(self):
        super().__init__()
        self.agent = RAGAgent(self)
        
        print("--- INITIALIZING AGENT NEURAL PATHWAYS ---")
        # Injecting isolated memories. Notice how only one is relevant to berries.
        self.agent.add_memory("I found a clean river to the north. Good for drinking.")
        self.agent.add_memory("Wolves hunt in packs near the eastern mountains. Avoid at night.")
        self.agent.add_memory("Last week, I ate red berries from a bush and got violently ill for two days. They are highly toxic.")
        print("------------------------------------------\n")

    def step(self):
        self.agent.step()

In [4]:
model = RAGModel()
model.step()

--- INITIALIZING AGENT NEURAL PATHWAYS ---
🧠 [MEMORY STORED] I found a clean river to the north. Good for drinking.
🧠 [MEMORY STORED] Wolves hunt in packs near the eastern mountains. Avoid at night.
🧠 [MEMORY STORED] Last week, I ate red berries from a bush and got violently ill for two days. They are highly toxic.
------------------------------------------


👀 [SITUATION ENCOUNTERED] I am starving. I see a large bush covered in bright red berries. What should I do?
🔍 [RECALLED MEMORY] Last week, I ate red berries from a bush and got violently ill for two days. They are highly toxic.
🤖 [DECISION] Do not eat the berries.
